# 02.5 Ufuncs, Reductions & Missing Data

> **Prerequisites:** 02.2 (NaT and accumulator dtypes) · 02.3 (keepdims and the declared
> `(n, 1)`) · 02.4 (`np.add.at`; boundary-normalised status) · 01.4/01.6 (label censoring
> and survivorship — this notebook is where they reach the arithmetic)
> **What you'll learn:**
> - Read any `np.<ufunc>.<method>` from two parts: one elementwise core, five method hats — reduce, accumulate, outer, at
> - Keep axis semantics straight on real 2-D data: the axis you sum is the axis you lose, and `keepdims` is the broadcast bridge
> - Predict what one NaN does to a plain reduction, and what the `nan*` family *actually* answers when it "fixes" that
> - Show that ignoring missing values is a population decision wearing a syntax choice, using a censoring experiment with a known truth
> - Ship rollups that carry their denominator: statistic + coverage + verdict, with WITHHOLD as a first-class output
> **Level:** Beginner · **Series:** 02 NumPy & Vectorized Computing

> ⚡ **Wednesday 2026-09-02, 07:20** — the "average days to payment" dashboard is blank.
> Not the recent months: every month, back to January 2019. The nightly job is green, the
> exports hash clean, and the refactor that shipped two days ago was the *right* one — it
> stopped silently dropping unresolved invoices, exactly as last quarter's survivorship
> post-mortem demanded. The cause: the dashboard's arithmetic was never asked whether it
> could survive honesty.


## Concept
### Plain-English Explanation

Every aggregate this series has printed — sums, means, counts — is a **reduction**: an
operation that folds many values into fewer. This notebook is about the machinery under
those folds and the one question every fold quietly answers on your behalf: *what is the
denominator?* When the data is complete, nobody notices the question. The moment values are
missing — and in PayFlow they are structurally missing, because recent invoices have not
resolved yet — every possible answer is a policy with consequences.

numpy offers three policies. Plain reductions propagate: one NaN anywhere makes the answer
NaN, which is honest and unusable — the cold open's blank wall. The `nan*` family ignores:
compute on whatever is present, which is usable and quietly changes the denominator per
month — the hotfix that made collections look faster every week. And the third policy is
the one no function ships: report the statistic *with* its denominator, and refuse to
report when too little of the population is present. That third policy is Stage C, and the
whole notebook builds toward why it has to be written by hand.

### Technical Explanation

**The ufunc anatomy.** A ufunc is one elementwise core — `np.add`, `np.maximum`,
`np.logical_or` — wearing method hats: `reduce` folds to one (`.sum()` is
`np.add.reduce`), `accumulate` keeps the running fold, `outer` fans every pair (02.3's
declared fan-out as a method), and `at` scatters unbuffered (02.4's duplicate-target fix).
⭐ **CRITICAL CONCEPT** — learning the four hats once buys the whole family:
`np.maximum.accumulate` is a running high-water mark, `np.logical_or.reduce` is `any`,
and every reduction keyword below (`axis`, `keepdims`, `where`, `initial`, `dtype` — the
02.2 accumulator lever) applies uniformly.

**Axis semantics.** On a 2-D pivot of shape `(92 months, 4 statuses)`, `sum(axis=0)`
collapses months into per-status totals `(4,)`; `sum(axis=1)` collapses statuses into
per-month totals `(92,)`. The axis named is the axis *removed* — and
`keepdims=True` retains it as length 1, handing 02.3 exactly the declared `(92, 1)` it
needs to broadcast shares against row totals.

**NaN through the fold.** IEEE NaN is absorbing: any arithmetic touching it yields NaN,
so a plain reduction over 288,040 values returns NaN if even one of the 12,529 unresolved
invoices is present — 4.3% missing blanks 100% of the answer. The `nan*` functions
substitute the policy "ignore the missing" — and carry their own traps: `nansum` of an
all-NaN slice returns a confident `0.0` while `nanmean` returns NaN with a warning, so
two functions from the same family disagree about what *no data* means. Neither ever
answers the question that matters: is the surviving subset representative? For
days-to-payment it demonstrably is not — invoices resolved early are the fast payers, so
a mean over "whoever has paid" is biased fast by construction.

### Mental Model

A reduction is a fraction, and every missing-data policy is a decision about its
denominator: propagate (refuse the fraction), ignore (shrink it silently), or report it
(carry numerator, denominator and a verdict together). If you cannot say what your
denominator was, you do not have a statistic — you have a mood.


## How It Works

```text
  ufunc = one elementwise core + five method hats
    np.add(a, b)   add.reduce   add.accumulate   add.outer      add.at
    elementwise    fold -> 1    running fold     every pair     unbuffered scatter

  axis semantics on pivot[92 months, 4 statuses]:
    sum(axis=0)  collapse DOWN the months    -> (4,)    per-status totals
    sum(axis=1)  collapse ACROSS statuses   -> (92,)   per-month totals
    sum(axis=1, keepdims=True)              -> (92, 1)  broadcast-ready (02.3)
                 the axis you SUM is the axis you LOSE

  one month's days_late through three policies:
    [ 3.0 | nan | 1.0 | 7.0 | nan | ... ]
       -- mean -->     nan                    honest, unusable: one NaN poisons all
       -- nanmean -->  mean of the resolved   usable, and the denominator just moved
       -- Stage C -->  (mean, coverage, verdict)   the denominator is a printed column

  nanmean's hidden assumption: the resolved subset is a fair sample.
  For payment delays it cannot be - being resolved EARLY means paying FAST -
  so the more a month is censored, the faster it falsely looks.
```

The bottom block is the incident's entire mechanism, and it is worth stating as a chain.
An invoice's `days_late` exists only after its first payment arrives, so recent months are
**right-censored**: 2026-08 has resolved only 21.2% of its invoices (01.4's gradient,
recomputed below). Which 21.2%? Not a random sample — the invoices that resolved within
days of issue, i.e. the fastest payers in the month. `nanmean` then averages exactly that
subset. The censoring is invisible in the output: the function returns one well-formed
float, the dashboard renders it, and the trend "improves" every week for the worst
possible reason. Propagating NaN at least *refused* to answer; ignoring NaN answers a
different question — "how late were the invoices that have already paid?" — while wearing
the label of the original one.

That reframing is the transferable lesson: `np.nanmean` is not "mean, but robust". It is
a *different estimator* with a hidden sampling assumption, and whether it approximates the
mean depends entirely on why the data is missing — a distinction series 09 will formalise
as missingness mechanisms, met here first as arithmetic.


## Hands-On Build
### Stage A — from scratch

Collapsed (library-API notebook): a reduction's raw mechanism — a loop with an
accumulator — was built and priced in 02.2's float32 running loop; the ufunc machinery
*is* the library surface under study here.

### Stage B — idiomatic

First the lane, joined to its payments so that missingness is real rather than injected:
every unresolved invoice becomes an honest NaN.


In [1]:
# Load the committed lab module; it owns the stdlib parse and join: M1 duplicates
# skipped and counted, M6 status lowered, M8 dual timestamp formats, M13 first payment.
import importlib.util
import sys
from pathlib import Path

import numpy as np

LAB = Path.cwd() / "_lab" / "lab_02.5_reductions.py"
spec = importlib.util.spec_from_file_location("lab_02_5", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_02_5"] = lab
spec.loader.exec_module(lab)

d = lab.load_lane()
print(f"invoices={len(d['days_late']):,} (skipped {d['n_dupes']} M1 duplicate rows)   "
      f"months={len(d['months'])}")
print(f"payments: {d['n_legacy_ts']:,} legacy DD/MM/YYYY timestamps (M8) parsed by "
      f"fallback")
print(f"unresolved invoices -> NaN days_late: {int(np.isnan(d['days_late']).sum()):,}")

invoices=288,040 (skipped 896 M1 duplicate rows)   months=92
payments: 43,165 legacy DD/MM/YYYY timestamps (M8) parsed by fallback
unresolved invoices -> NaN days_late: 12,529


Three loader numbers close loops opened in series 01. The 896 skipped rows are M1's
repost bug, counted here at the boundary rather than discovered downstream. The 43,165
legacy timestamps are exactly the payments 01.1 showed a naive single-format parse would
silently discard — parsed here by a two-format fallback, so the join loses nothing. And
the 12,529 NaNs are 01.2's unpaid invoices, kept in the frame this time: the survivorship
population 01.6 measured is now *present and marked* instead of silently absent. Honest
missingness is the raw material for everything below.

The machinery next — one ufunc, five hats.


In [2]:
lab.ufunc_anatomy(d)

  np.add is ONE elementwise core wearing five method hats:
    add(a, b)          elementwise    (every arithmetic op in this series)
    add.reduce(a)      fold to one   == a.sum(): 1,441,596 == 1,441,596  parity True
    add.accumulate     running fold  invoices issued, cumulative: first 4 months [29, 81, 167, 301], all-time 288,040
    add.outer          every pair     (02.3's declared fan-out, as a method)
    add.at             unbuffered scatter (02.4's duplicate-target fix)

  reduce, accumulate, outer, at - every ufunc ships all four, so the
  toolkit generalises: np.maximum.reduce is max, np.maximum.accumulate is
  a running high-water mark, np.logical_or.reduce is 'any', and so on.


The parity line is the demystification: `.sum()` *is* `np.add.reduce`, to the digit —
method syntax over the same fold. `accumulate` turns the same core into running state (the
cumulative issue counts will feed growth charts in series 28), and `outer` and `at` are
02.3's and 02.4's mechanisms surfacing as methods of every ufunc rather than one-off
functions. The generalisation is the point: nothing new needs learning for
`np.maximum.accumulate` (a running high-water mark for a worst-delay tracker) or
`np.logical_or.reduce` — the hats transfer.

Now the axis question, on a pivot big enough to punish guessing.


In [3]:
lab.pivot_axis_semantics(d)

  pivot[month, status]: shape (92, 4)  (92 months x statuses ['disputed', 'overdue', 'paid', 'written_off'])
                  disputed     overdue        paid written_off
    2026-06            119         121       6,938          29
    2026-07            100       1,049       6,128          24
    2026-08             27       5,664       1,537           6

  sum(axis=0) collapses MONTHS   -> shape (4,): {np.str_('disputed'): 4298, np.str_('overdue'): 6848, np.str_('paid'): 275797, np.str_('written_off'): 1097}
  sum(axis=1) collapses STATUSES -> shape (92,): per-month totals, e.g. 2026-08 = 7,234
  sum(axis=1, keepdims=True)     -> shape (92, 1): ready to broadcast as shares = pivot / row_totals (02.3's declared (n,1))
  worked example: 2026-08 status mix is 78% 'overdue' - the axis you SUM is the axis you LOSE
  grand total, both orders: 288,040 == 288,040 == invoices loaded


The pivot is built by 02.4's `np.add.at` working in 2-D — the index is a *tuple* of
arrays, one per axis, and 288,040 (month, status) pairs scatter-accumulate into 368 cells
without a loop. Then the two reductions: `axis=0` collapses months and answers "how many
invoices ever end in each status" — 275,797 paid against 4,298 disputed and 1,097 written
off, the same outcome mix 01.6's survivorship audit counted; `axis=1` collapses statuses
into monthly volumes. The mnemonic printed under the table is the one that sticks: **the
axis you sum is the axis you lose.**

⚠️ Two supporting readings. `keepdims=True` returns `(92, 1)` instead of `(92,)` — that is
02.3's *declared* column, produced at the reduction that needs it, so
`pivot / pivot.sum(axis=1, keepdims=True)` broadcasts shares without a detached reshape.
And the worked example is a data point to hold: 2026-08's status mix is 78% `overdue` —
recent months are dominated by not-yet-resolved outcomes, which is the censoring gradient
about to become the incident.

The missing-value algebra itself:


In [4]:
lab.nan_algebra(d)

  days_late: 288,040 invoices, 12,529 unresolved -> NaN (4.3% of the lane)
  NaN algebra: nan == nan -> False; membership needs np.isnan (and np.isnat for NaT, 02.2)
  propagation: x.sum() = nan   x.mean() = nan   one NaN poisons any plain reduction
  TRAP: nansum of an ALL-NaN slice = 0.0 - a confident,
  silent zero - while nanmean gives nan with RuntimeWarning('Mean of empty slice'):
  the two nan-functions disagree about what 'no data' means
  0/0 under np.errstate(invalid='ignore') -> nan with the RuntimeWarning
  suppressed; unguarded it warns - decide loud-or-quiet per call site,
  and narrowly: a blanket ignore hides every future invalid op too

  the mask-explicit spelling: x.sum(where=~isnan, initial=0) / count
    = 5.232   vs np.nanmean(x) = 5.232   same number, but the mask and the count are now VISIBLE variables

  label coverage by issue month (the tail of 01.4's censoring gradient):
    2026-05  coverage 98.2%
    2026-06  coverage 96.2%
    2026-07  coverage 83.8%
    

Four behaviours worth separating. **Propagation**: one NaN anywhere and both
`sum()` and `mean()` over the whole lane return NaN — absorbing arithmetic, no partial
credit. **The family disagreement**: `nansum` of an all-NaN slice is a confident, silent
`0.0` while `nanmean` is NaN plus a `Mean of empty slice` warning — so "a month with no
data" becomes zero revenue under one spelling and a warning under another, and any
dashboard mixing the two invents a difference between identical situations.
**`errstate`**: floating-point events like `0/0` are controllable per scope — but keep the
scope tight; a blanket `invalid="ignore"` also silences the *next* bug's warning.
**The mask-explicit spelling**: `x.sum(where=~isnan(x), initial=0)` divided by the mask's
count reproduces `nanmean` exactly (5.232 days, both spellings) while making the mask and
the count *named variables* — which is precisely what Stage C needs, because the count is
about to become the star of the show.

The coverage table at the end is the plot: 98.2% → 96.2% → 83.8% → **21.2%** across the
last four months. Hold it against the pivot's 78%-overdue reading and the incident is
fully staged.

### Stage C — production

The rollup as a contract: every monthly statistic ships with its denominator and a
verdict, and months below the coverage floor say WITHHOLD instead of a number.


In [5]:
lab.contract_demo(d)

  month         mean  coverage   verdict
  2026-03       7.36     98.0%   report
  2026-04       7.23     97.8%   report
  2026-05       6.95     98.2%   report
  2026-06       6.69     96.2%   report
  2026-07       5.02     83.8%   WITHHOLD (< 90% resolved)
  2026-08       2.34     21.2%   WITHHOLD (< 90% resolved)
  [PASS] reported months meet the coverage floor
  [PASS] withheld months are exactly the complement
  [PASS] means agree with nanmean where defined
  [PASS] no plain-mean NaN leaks into a reported month

  the verdict column is the artifact: a dashboard that can say WITHHOLD
  cannot be silently biased by its own denominator - and the coverage floor
  is a reviewable number, not a reflex buried in a nan-function.


Read the table's last two rows first: 2026-07 and 2026-08 produce numbers — 5.02 and
2.34 — and the contract **refuses to report them**, because 83.8% and 21.2% coverage fail
the 90% floor. That refusal is the artifact. `robust_monthly_mean` computes the same
resolved-only mean `nanmean` would (asserted in the PASS lines), but returns it as a
triple — statistic, coverage, verdict — with the floor as a named, reviewable constant
rather than a reflex buried in a function name. The four PASS lines pin the contract:
reported months meet the floor, withheld months are exactly the complement, parity with
`nanmean` where defined, and no propagated NaN leaks into a reported cell.

## Evaluation

Assertion-shaped, as throughout the series (guide §2). Captured layers: the
**reduce/sum parity** line (method and shorthand agree to the digit); the **pivot's
closure check** (grand total equals invoices loaded whichever axis order does the
collapsing — a two-way reconciliation in 01.2's spirit); the **spelling parity** between
`nanmean` and the mask-explicit `where=`/count form (5.232 both ways); the **controlled
censoring experiment** in the scenario below, which measures the bias against a known
truth rather than asserting it; and the **four Stage C PASS lines**. Deterministic
throughout — any flipped line is a semantics change, not noise.


## Design Patterns / Tradeoffs

**`nan*` functions versus explicit masks.** The `nan*` family is one call and reads
cleanly, but it hard-codes a policy (ignore) and hides the denominator; its members even
disagree at the empty edge (`nansum` → 0.0, `nanmean` → NaN + warning). The explicit form
— build `mask = ~np.isnan(x)` once, then `sum(where=mask)`, `count_nonzero(mask)` — is
two lines longer and makes the policy and the denominator inspectable, testable
variables. Use `nan*` for throwaway analysis where the missingness is understood and
immaterial; use explicit masks the moment a number leaves your notebook for a dashboard,
a report or a model, because that is when the denominator starts needing an audit trail.

**Propagate versus ignore versus verdict, as a *system* choice.** Propagation is the
right default inside computations — a NaN reaching a place it shouldn't is a bug
announcing itself, which is why plain `mean` refusing was the most honest day the
dashboard ever had. Ignoring is right when missingness is verifiably unrelated to the
quantity (a sensor dropping random packets) — and wrong, specifically and directionally,
when missingness *is* the quantity, as with censored payment delays where absence means
slowness. The verdict pattern subsumes both for anything user-facing: compute like
`ignore`, publish like `propagate` when coverage fails, and print the coverage either
way. Its cost is honest: someone must own the floor (90% here), and a dashboard must be
allowed to say "insufficient data" — an organisational fight, not a technical one.

**`errstate` scoping.** Floating-point noise control belongs at the *narrowest* scope
that owns the intent: the Stage C division wraps exactly one line where `0/0 → NaN` is
the designed outcome. Module-wide suppression is how the next incident's warning dies
unread; conversely, promoting warnings to errors (`np.errstate(invalid="raise")`) in
tests turns accidental NaN manufacture into a stack trace — cheap CI armour.

**Recommendation for PayFlow:** explicit masks and named coverage for every published
aggregate; the verdict triple (statistic, coverage, verdict) as the standard rollup
return shape; `nan*` allowed in exploration, banned by review from dashboards;
`errstate` blocks no wider than one expression, and `invalid="raise"` in the test suite.


## Production Scenario
### Symptoms

**Wednesday 2026-09-02, 07:20.** Two days ago, engineering shipped the fix from last
quarter's survivorship post-mortem (01.6): the analytics loader stopped inner-joining
payments and now keeps unresolved invoices, with `days_late` as NaN. The nightly rollup
job itself was untouched.

- **07:20** — the "average days to payment by month" panel is blank for *all 92 months*,
  January 2019 through August 2026. Not zero — empty cells, where the JSON carries `null`.
- The job is green; the exports hash identical to Monday's manifest; row counts are up
  slightly (the previously-dropped unresolved invoices, as intended by the refactor).
- **Phase two, 09:40** — a hotfix replaces `mean` with `np.nanmean`. Panels return.
  The trend now shows collections accelerating dramatically: 6.95 days in May 2026,
  5.02 in July, 2.34 in August. A congratulatory message appears in the collections
  channel before lunch.
- The finance lead asks one question that stops the celebration: "August isn't over for
  most of those invoices — whose average is that?"


In [6]:
lab.incident(d)

  era 1 (inner join, resolved only): mean days_late per month; e.g. 2025-03 = 4.74, stable and quietly survivorship-biased (01.6)
  era 2 (honest NaNs + plain mean): 92 of 92 monthly panels read NaN
    -> one unresolved invoice anywhere in a month blanks that month; the
       dashboard goes empty back to 2019-01, job green throughout


  era 3 (the nanmean hotfix): panels return, and the newest read 2.34 days (2026-08) vs 6.95 (2026-05) - collections looks suddenly faster

  controlled censoring of 2025-03 (its truth is fully resolved):
    true mean days_late            :    4.74  (coverage 100%)
    observed 30 days after month-end:    2.78  (coverage 86%)
    nanmean under censoring reads 1.96 days FASTER than
    truth - it silently averages whoever has already paid, and fast payers
    pay first.
    The era-3 dashboard is this experiment, live, on every recent month.


### Diagnosis

Walking the ladder in its data-pipeline form, naming what each signal eliminated:

1. **Alert** — every panel blank at once, immediately after a loader change. Candidate
   causes: the loader broke the data, the rollup broke on the data, or the rollup was
   never compatible with the data's new honesty.
2. **Job logs and input checks** — green and clean; hashes match; the row-count increase
   is exactly the previously-dropped unresolved invoices. The loader did precisely what
   the post-mortem asked. Data eliminated; the interaction is the suspect.
3. **Value inspection** — the monthly means are NaN for 92 of 92 months, and
   `np.isnan(days_late)` counts at least one unresolved invoice in *every* month of the
   lane's history (12,529 in total — disputes and write-offs live in old months too,
   which is 01.6's survivorship population showing up as arithmetic). Plain `mean`
   propagates, so one honest NaN per month blanks the century. Mechanism one, named.
4. **The hotfix, audited** — `nanmean` restores numbers by silently changing each month's
   denominator from "invoices issued" to "invoices resolved so far". The coverage
   gradient (98.2% → 21.2% over the last four months) means recent panels average an
   ever-faster subset. Mechanism two, named — but so far only as an argument.
5. **The controlled experiment** — the cell above censors March 2025, a month whose truth
   is fully resolved: true mean 4.74 days; observed through a 30-days-after-month-end
   cutoff, `nanmean` reads 2.78 on 86% coverage — **1.96 days fast**, against ground
   truth, with no drift or regime change involved. The bias is measured, not asserted.
   August's 2.34 is this experiment running live at 21.2% coverage.
6. **Version diff, complete** — era 1's inner join had the *same* resolved-only
   denominator all along; it just applied it invisibly, upstream, forever. The refactor
   didn't create the bias — it surfaced a question the rollup had never been asked.

### Root Cause

The rollup's arithmetic embedded a denominator policy nobody had chosen: plain `mean`
refused honest missingness outright (one unresolved invoice per month blanked all 92
panels), and the `nanmean` hotfix answered a different question — "how late were the
invoices that have already paid?" — whose bias grows with censoring, making the most
censored months look fastest. Both failures trace to the same absence: no statistic
carried its coverage.

### Fix

**Mitigation now.** Freeze the trend panel with an annotation, and republish from the
Stage C rollup: report months at ≥90% coverage (through 2026-06), WITHHOLD 2026-07 and
2026-08 with their coverage printed in the cell — "insufficient data" is a publishable
result.

**Permanent fix.** `robust_monthly_mean` becomes the only sanctioned rollup: statistic +
coverage + verdict, floor a named constant under review, parity-with-`nanmean` and
no-NaN-leak assertions in the suite. The dashboard schema gains a WITHHOLD state so
honesty has somewhere to go.

### Prevention

- **Every published aggregate carries its denominator.** A mean without its coverage is
  unauditable; the triple costs one `bincount`.
- **The censoring experiment joins the test suite**: censor a fully-resolved month,
  require the withheld verdict to fire before the measured bias exceeds the floor's
  implied tolerance — the bias check with a known truth, run on every change.
- **Trend improvements trigger the same scrutiny as regressions.** "Collections
  accelerating" survived four hours because good news is under-audited; 01.1's
  green-dashboard incident said the same.
- **`invalid="raise"` in tests** — a NaN manufactured where none is expected should be a
  stack trace in CI, not a blank panel in production.


## Common Pitfalls

⚠️ **Treating `nanmean` as "mean, but robust".** It is a different estimator with a
sampling assumption — the missing are like the present. Under censoring that is false by
construction, and the bias grows exactly where coverage falls.

⚠️ **`nansum` of nothing is zero.** An all-NaN slice sums to a confident `0.0` — a month
with no data becomes a month with zero revenue. Its own sibling `nanmean` disagrees (NaN
plus a warning). Check emptiness explicitly; never let a nan-function define "no data".

**Blanket `errstate` suppression.** `invalid="ignore"` around a whole module silences the
warning that would have flagged the *next* bug. One expression per suppression, with the
intent commented.

**Averaging ratios instead of ratio-of-sums.** `(a / b).mean()` weights every row
equally; `a.sum() / b.sum()` weights by size. Both are "the average" in a meeting; they
diverge whenever sizes vary, so name which one the metric means.

**Losing the axis you meant to keep.** `sum(axis=0)` on a `(months, statuses)` pivot
answers a different business question than `axis=1`; the mnemonic — the axis you sum is
the axis you lose — plus a shape assert beats re-deriving under incident pressure.

**Trusting a green trend.** The biased dashboard looked like *good* news, and good news
gets four hours of grace that bad news never would. Symmetric scepticism is a prevention
control, not a temperament.

**Forgetting `initial=` with `where=`.** A fully-masked reduction without `initial` has
nothing to start from and raises; with `initial=0` it returns the identity — decide which
of those your call site wants before the empty month arrives.


## Interview Questions

1. **Derive this.** A month has n invoices, r resolved, and resolution order correlates
   negatively with payment delay. Show the sign of `nanmean`'s bias for the month's true
   mean delay, and what happens to it as r/n falls. *Answer shape:* nanmean averages the
   resolved subset; if the unresolved have stochastically larger delays (censoring:
   unresolved means not-yet-paid), the resolved subset's mean is a lower bound, so the
   estimator is biased fast, and the bias is monotone in the censored share — smallest
   coverage, largest understatement.
2. **Design this.** Design the monthly-rollup contract for a metric whose inputs resolve
   over weeks. *Answer shape:* statistic + coverage + verdict per month; a named coverage
   floor under review; WITHHOLD as a publishable dashboard state; parity tests against
   the naive estimator where defined; a censoring test on a fully-resolved month with a
   bias tolerance tied to the floor.
3. **Debug this.** After a data-honesty refactor, a dashboard goes fully blank; a
   one-line hotfix restores it and the trend immediately improves implausibly. Walk both
   phases. *Answer shape:* phase one — plain reductions propagate the newly-honest NaNs,
   so any month containing one blanks; phase two — the nan-function restores numbers by
   shrinking each month's denominator to the resolved subset, so censored months average
   fast payers; confirm with a known-truth censoring experiment, then ship the
   denominator-carrying rollup.
4. Why do `np.nansum([])`-shaped and `np.nanmean([])`-shaped calls disagree, and what
   should production code do about the empty case? *Answer shape:* sum's identity is 0 so
   nansum returns 0.0 silently; mean has no identity, so nanmean yields NaN with a
   warning. Production code checks the count explicitly and routes "no data" to an
   explicit state rather than letting either function improvise.
5. What do `where=` and `initial=` buy over the `nan*` family? *Answer shape:* the mask
   and the denominator become named variables — inspectable, testable, reusable for
   coverage — and the empty-reduction behaviour is chosen (`initial`) rather than
   inherited; the nan-functions are the same computation with the policy hard-coded and
   the count discarded.
6. `pivot.sum(axis=1, keepdims=True)` — why keepdims, in one sentence each of mechanism
   and intent? *Answer shape:* mechanism — the collapsed axis survives as length 1, so
   the result is `(92, 1)` and broadcasts against `(92, 4)` by 02.3's rule; intent — the
   column shape is *declared at the reduction that creates it*, not manufactured later by
   a detached reshape (02.3's incident).
7. When is propagation the behaviour you want? *Answer shape:* inside computations, as a
   tripwire — a NaN arriving where none belongs is a bug that should surface, and
   `errstate(invalid="raise")` in tests makes it a stack trace; suppression and ignoring
   are for boundaries where the policy has been explicitly decided.


## Key Takeaways

- One elementwise core, five hats — reduce, accumulate, outer, at — and the keywords
  (`axis`, `keepdims`, `where`, `initial`, `dtype`) apply across the whole ufunc family.
- The axis you sum is the axis you lose; `keepdims=True` is how a reduction declares the
  `(n, 1)` it hands to broadcasting, at the site that creates it.
- One NaN poisons any plain reduction — 4.3% missing blanked 100% of the dashboard — and
  that refusal is the honest behaviour, not the bug.
- The `nan*` family doesn't fix missingness; it silently changes the denominator — and
  under censoring that estimator is biased in a known direction, measured here at 1.96
  days fast on a month with known truth.
- `nansum` and `nanmean` disagree about "no data" (confident 0.0 versus NaN-plus-warning):
  check emptiness yourself before either answers for you.
- Prefer the mask-explicit spelling for anything published: the mask and the count become
  variables, and the count is the coverage your dashboard owes its readers.
- Ship rollups as statistic + coverage + verdict with a named floor; WITHHOLD is a
  publishable result, and it is the only honest one at 21.2% coverage.
- Scope `errstate` to single expressions, and run tests with `invalid="raise"` — NaN
  manufacture should be a CI stack trace, not a production blank.


## Related

**Backward**

- **02.2 Numerical Dtypes & Promotion** — NaT's NaN discipline and the `dtype=` accumulator
  keyword both resurface inside every reduction here.
- **02.3 Broadcasting** — `keepdims` produces the declared `(n, 1)` at the reduction;
  the shares computation is that notebook's legitimate fan-out in daily use.
- **02.4 Indexing & Selection** — `np.add.at` scales to 2-D pivots; the mask-explicit
  reductions reuse its predicate discipline.
- **01.4 / 01.6** — the censoring gradient and the survivorship population, met there as
  modelling lessons, arrive here as arithmetic on the same 12,529 invoices.

**Forward**

- **02.6 Random Numbers & Reproducible Sampling** — the resample machinery under any
  claim that a subset is "representative".
- **02.7 Memory Layout & Performance** — `out=` and reduction order as allocation levers.
- **09.1 Data Cleaning & Pre-processing** — missingness mechanisms (MCAR/MAR/MNAR) done
  properly; today's censoring is the MNAR case with a business face.
- **28.1 Time Series Forecasting** — right-censored recent history is the standing
  condition of every forecast target.
